# VietHandOCR Part 2: Baseline Evaluation

Welcome to **Part 2** of the VietHandOCR pipeline.
- **Previous Notebook**: [Part 1: Data Preparation & EDA](./01_Data_Preparation_and_EDA.ipynb)
- **Next Notebook**: [Part 3: Digital Image Processing (DIP) Pipeline](./03_Digital_Image_Processing.ipynb)

## Introduction
This notebook establishes the baseline performance of the standard `vgg_transformer` model on the writer-independent dataset splits. By measuring the Zero-shot performance (without any prior task-specific fine-tuning or image processing), we create a benchmark to evaluate the quantitative impact of the upcoming Digital Image Processing (DIP) heuristics and domain-adaptation fine-tuning.

## Objectives
1. **Inference**: Perform Zero-shot inference across all text levels (word, line, paragraph).
2. **Metrics Calculation**: Compute Character Error Rate (CER), Word Error Rate (WER), Exact Match accuracy, and BLEU-2 scores.
3. **Error Analysis Logging**: Export detailed step-by-step evaluation logs and aggregated metrics for further analysis.


In [ ]:
!pip install -q vietocr jiwer nltk Pillow==10.2.0


In [ ]:
import os
import torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import jiwer
import nltk
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt', quiet=True)

from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

def calculate_metrics(predictions, targets):
    '''Calculates CER, WER, Exact Match, and BLEU.'''
    cer = jiwer.cer(targets, predictions)
    wer = jiwer.wer(targets, predictions)
    exact_match = sum(1 for p, t in zip(predictions, targets) if p == t) / len(targets)
    
    bleu_scores = []
    for p, t in zip(predictions, targets):
        reference = [nltk.word_tokenize(t.lower())]
        candidate = nltk.word_tokenize(p.lower())
        if len(candidate) == 0:
            bleu_scores.append(0.0)
            continue
        try:
            bleu = nltk.translate.bleu_score.sentence_bleu(reference, candidate, weights=(0.5, 0.5))
            bleu_scores.append(bleu)
        except Exception:
            bleu_scores.append(0.0)
            
    avg_bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0.0
    
    return {"CER": cer, "WER": wer, "Exact Match": exact_match, "BLEU": avg_bleu}


In [ ]:
def evaluate_baseline(predictor, test_txt_path, images_base_dir, level_name='all'):
    with open(test_txt_path, 'r', encoding='utf-8') as f:
        lines = f.read().strip().split('\n')
        
    targets = []
    predictions = []
    results_detail = []
    
    print(f"  [1/3] READING INPUT DATA...")
    print(f"    - File input: {test_txt_path}")
    print(f"    - Base Image Dir: {images_base_dir}")
    print(f"    - Total data records (images): {len(lines)}")
    print(f"  [2/3] STARTING ZERO-SHOT INFERENCE...")
    for line in tqdm(lines):
        if not line.strip(): continue
        parts = line.split('\t')
        if len(parts) != 2: continue
            
        rel_img_path, ground_truth = parts
        ground_truth = ground_truth.strip()
        if not ground_truth: continue # Skip empty ground truth
        
        full_img_path = os.path.join(images_base_dir, rel_img_path)
        
        try:
            with Image.open(full_img_path) as img:
                pred = predictor.predict(img).strip()
            
            targets.append(ground_truth)
            predictions.append(pred)
            
            results_detail.append({
                'image_path': rel_img_path,
                'ground_truth': ground_truth,
                'prediction': pred,
                'is_exact_match': ground_truth == pred
            })
        except Exception as e:
            print(f"Error processing {full_img_path}: {e}")
            
    if not targets:
        print("No valid targets found. Skipping metrics calculation.")
        return {}, pd.DataFrame()
        
    metrics = calculate_metrics(predictions, targets)
    print("\n" + "="*40)
    print(f"🏆 BASELINE RESULTS (Zero-shot) - LEVEL: {level_name.upper()}")
    print("="*40)
    for k, v in metrics.items():
        print(f"{k:<15}: {v:.4f}")
    print("="*40)
    
    df_results = pd.DataFrame(results_detail)
    df_results.to_csv(f'baseline_error_analysis_{level_name}.csv', index=False, encoding='utf-8')
    output_csv = f'baseline_error_analysis_{level_name}.csv'
    print(f"  [3/3] EXPORTING EVALUATION RESULTS...")
    print(f"    - Detailed error analysis output path: {os.path.join(os.getcwd(), output_csv)}")
    print(f"    - Successfully saved {len(df_results)} prediction records.")
    
    return metrics, df_results


In [ ]:
import os
import glob
import gc
import torch
import pandas as pd
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

# Robust Dataset Path Resolution
def get_dataset_path():
    kaggle_input = '/kaggle/input'
    local_input = 'VietHandOCR_Datasets'
    if os.path.exists(kaggle_input):
        # Look for the structure containing UIT_HWDB_word or UIT_HWDB_line
        for root, dirs, files in os.walk(kaggle_input):
            if any('UIT_HWDB_' in d for d in dirs):
                return root
        return kaggle_input
    return local_input

base_dataset_path = get_dataset_path()

# Locate all generated test_*.txt and val_*.txt evaluation sets in Kaggle /input or local directory
eval_files = {}
search_dirs = ['../input', '.']
for sdir in search_dirs:
    if os.path.exists(sdir):
        for root, dirs, files in os.walk(sdir):
            # Skip hidden and repository directories to prevent erroneous matching
            if '.git' in root or '.ipynb_checkpoints' in root: continue
            for file in files:
                if (file.startswith('test_') or file.startswith('val_')) and file.endswith('.txt'):
                    # Extract split_level name, e.g. test_line.txt -> test_line
                    name = file.replace('.txt', '')
                    if name not in eval_files: # Prefer the first found path
                        eval_files[name] = os.path.join(root, file)

if not eval_files:
    print("Warning: No evaluation files (test_*.txt or val_*.txt) found. Please run 01_Data_Preparation_and_EDA.ipynb first to generate them.")
else:
    print(f"Found dataset at: {base_dataset_path}")
    print(f"Found {len(eval_files)} evaluation files: {list(eval_files.keys())}")
    
    print("\n" + "="*50)
    print("🤖 LOADING MODEL (vgg_transformer)")
    print("="*50)
    
    # Load the standard VGG19 + Transformer model for the baseline evaluation
    config = Cfg.load_config_from_name('vgg_transformer')
    
    config['device'] = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    predictor = Predictor(config)
    print(f"Loaded predictor successfully on {config['device']}!")
    
    all_metrics = {}
    for name, txt_path in eval_files.items():
        print(f"\n{'='*50}")
        print(f"🚀 EVALUATING: {name.upper()}")
        print(f"{'='*50}")
        
        metrics, df_results = evaluate_baseline(predictor, txt_path, base_dataset_path, level_name=name)
        if metrics:
            all_metrics[name] = metrics
            
        # Free memory after each evaluation run to respect Kaggle constraints
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    print(f"\n{'*'*50}")
    print(f"📊 SUMMARY OF ALL EVALUATIONS (Zero-shot VGG19)")
    print(f"{'*'*50}")
    
    summary_records = []
    for name, metrics in all_metrics.items():
        print(f"--- Split: {name.upper()} ---")
        row = {'Split': name.upper()}
        for k, v in metrics.items():
            print(f"  {k:<12}: {v:.4f}")
            row[k] = v
        summary_records.append(row)
    print(f"{'*'*50}")
    
    # Export global summary CSV
    summary_df = pd.DataFrame(summary_records)
    summary_csv_path = 'baseline_summary_metrics.csv'
    summary_df.to_csv(summary_csv_path, index=False, encoding='utf-8')
    print(f"\n✅ EXPORTED SUMMARY METRICS: {os.path.join(os.getcwd(), summary_csv_path)}")
